# Investment Analysis for Equity Portfolio

## Project Overview

This project analyzes the performance and risk of an equity portfolio using historical market data.

The portfolio consists of:
- JPMorgan Chase (JPM)
- Morgan Stanley (MS)
- Bank of America (BAC)

The portfolio is compared with the **S&P 500** as the benchmark.

### Analysis Covered

- Simple returns
- Log returns
- Portfolio return
- Annualized return
- Volatility
- Beta
- CAPM Alpha
- Sharpe Ratio
- Sortino Ratio
- Maximum Drawdown
- Calmar Ratio
- Treynor Ratio
- Value at Risk (VaR)
- Expected Shortfall

**Purpose:** This is an educational quantitative-finance project to understand portfolio performance and risk measurement.

**Data source:** Yahoo Finance via `yfinance`.


In [ ]:
# !pip install yfinance

In [ ]:
# Import all the packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import yfinance as yf

In [ ]:
# JPM: JP Morgan
# MS: Morgan Stanley
# BAC: Bank of America

In [ ]:
# Download Stock Data from Yahoo Finance

tickers = ['JPM', 'MS', 'BAC']
start_date = '2024-01-01'
end_date = '2025-01-01'

data = yf.download(tickers, start = start_date, end = end_date)['Close']
data

In [ ]:
# Visualize our Stocks Data
px.line(data, x = data.index, y = data['BAC'], title = 'Closing Price of BAC')

In [ ]:
# Visualize our Stocks Data
px.line(data, x = data.index, y = data['JPM'], title = 'Closing Price of JPM', color_discrete_sequence= ['red'])

In [ ]:
# Visualize our Stocks Data
px.line(data, x = data.index, y = data['MS'], title = 'Closing Price of MS', color_discrete_sequence = ['green'])

In [ ]:
# Visualize All our Stocks Data
px.line(data, title = 'Closing Price of JPM, MS & BAC').show()

In [ ]:
data.head()

In [ ]:
# Calculate Simple Return => (St - St-1)/(St-1) or (St/St-1) - 1

simple_returns = data.pct_change()
simple_returns = simple_returns.dropna() # Drop nan values
simple_returns

In [ ]:
# Calculate Log Return => ln (St/St-1)

log_returns = np.log(data/data.shift(1))
log_returns = log_returns.dropna() 
log_returns

In [ ]:
# Portfolio Simple Returns (Rp = w1r1 + w2r2 + w3r3) => [w1 w2 w3].[r1 r2 r3]

weights = np.array([1/3, 1/3, 1/3])
portfolio_simple_returns = simple_returns.dot(weights)
portfolio_simple_returns

In [ ]:
# Portfolio Log Returns (Rp = w1r1 + w2r2 + w3r3) => [w1 w2 w3].[r1 r2 r3]
weights = np.array([1/3, 1/3, 1/3])
portfolio_log_returns = log_returns.dot(weights)
portfolio_log_returns

In [ ]:
# Annualize Simple Return of our Portfolio - Simple Return => (1 + Avg(Returns))^252 - 1
annualized_simple_return = ((1 + portfolio_simple_returns.mean()) ** 252) - 1
annualized_simple_return

In [ ]:
# Annualize Log Return of our Portfolio - Log Return => Avg Returns * 252
annualized_log_return = portfolio_log_returns.mean() * 252
annualized_log_return

In [ ]:
# Volatility
daily_volatility = np.std(portfolio_simple_returns)
annual_volatility = daily_volatility * np.sqrt(252)
annual_volatility

In [ ]:
# Download SPY Data (Benchmark)
start_date = '2024-01-01'
end_date = '2025-01-01'
benchmark = yf.download('^GSPC', start = start_date, end = end_date)['Close']
benchmark = benchmark.pct_change() # Simple Returns for my S&P 500
benchmark = benchmark.dropna()
benchmark

In [ ]:
# Calculate Beta
# Beta = Covariance(Rp, Rm)/Var(Rm) 
# Covariance: How 2 variable (Portfolio, Market) are moving together

portfolio_returns = portfolio_simple_returns.to_numpy().flatten()
benchmark_return = benchmark.to_numpy().flatten()

cov_matrix = np.cov(portfolio_returns, benchmark_return)
beta = cov_matrix[0,1]/cov_matrix[1,1]
beta

# Beta = 0.90 < 1 => Our portfolio is less volatile than the market
# For every 1% change in the market, out portfolio tend to change by ~0.90% in the same direction


In [ ]:
# Calculate Alpha => CAPM Formula to calculate Alpha
risk_free_rate = 0.07
alpha = (np.mean(portfolio_simple_returns) - risk_free_rate/252) - beta * (np.mean(benchmark) - risk_free_rate/252)
alpha = alpha * 252
alpha

# Alpha represent excess return of a portfolio relative to its benchmark
# Our portfolio outperformed the benchmark by 14.45%

In [ ]:
portfolio_simple_returns

In [ ]:
# Sortino Ratio
negative_returns = portfolio_simple_returns[portfolio_simple_returns < 0]
downside_deviation = np.std(negative_returns) # Daily Downside Std Dev
downside_deviation = downside_deviation * np.sqrt(252) # Annualized Downside Std Dev
downside_deviation

In [ ]:
# Sharpe Ratio = (Rp - Rf)/sigma => Total Volatility
sharpe_ratio = (annualized_simple_return - risk_free_rate)/annual_volatility
sharpe_ratio

# For every 1 unit of risk, the portfolio is generating 1.64 units of excess return
# This sharpe ratio is considered to be very good

In [ ]:
# Sortino Ratio = (Rp - Rf)/sigma(d) => Downside Volatility
sortino_ratio = (annualized_simple_return - risk_free_rate)/downside_deviation
sortino_ratio

# For every unit of downside risk, the portfolio is generating 2.58 units of excess return
# Sortino Ratio of 2.58 is considered to be really good

In [ ]:
# Calmar Ratio = Rp/Max Drawdown
# Max Drawdown => Cumulative Return

cumulative_simple_returns = (1 + portfolio_simple_returns).cumprod() 
max_drawdown = ((cumulative_simple_returns.cummax() - cumulative_simple_returns)/cumulative_simple_returns.cummax()).max()
max_drawdown

# Max Drawdown of 13% => It shows worst historical loss from peak to bottom

In [ ]:
cumulative_simple_returns = (1 + portfolio_simple_returns).cumprod()
running_max = cumulative_simple_returns.cummax()
drawdown = (running_max - cumulative_simple_returns)/running_max
drawdown.max()

In [ ]:
# Calmar Ratio 
calmar_ratio = annualized_simple_return/max_drawdown
calmar_ratio

In [ ]:
# Treynor Ratio (Rp - Rf)/Beta
treynor_ratio = (annualized_simple_return - risk_free_rate)/beta
treynor_ratio

In [ ]:
# Value at Risk (Historical Method) => potential loss in your portfolio
portfolio_value = 1000000
var_90 = np.percentile(portfolio_simple_returns, 10) * portfolio_value
var_95 = np.percentile(portfolio_simple_returns, 5) * portfolio_value
var_99 = np.percentile(portfolio_simple_returns, 1) * portfolio_value
var_99

In [ ]:
var_90

In [ ]:
var_95

In [ ]:
var_99

In [ ]:
# Conditional VaR / Expected Shortfall => E[Loss/Loss > VaR] => Avg of it (Returns > VaR Returns)
c_var = portfolio_simple_returns[portfolio_simple_returns <= np.percentile(portfolio_simple_returns, 5)].mean()
c_var = c_var * portfolio_value
c_var

In [ ]:
# Project: Investment Analysis on our Equity Portfolio

# Download the data from Yahoo Finance
# Visualized the Stocks/ Portfolio
# Simple Return
# Log Return
# Volatility Using Simple Return
# Volatility Using Log Return
# Alpha
# Beta
# Sharpe Ratio
# Sortino Ratio
# Calmar Ratio
# Treynor Ratio
# Max Drawdown
# Value at Risk
# Expected Shortfall

# Thank You

## Notes on Interpretation

- **Return** measures how much the portfolio gained or lost over the period.
- **Volatility** measures the variability of returns and is commonly used as a measure of risk.
- **Beta** measures the portfolio's sensitivity to movements in the S&P 500.
- **Alpha** represents performance relative to what would be expected from the portfolio's market exposure under CAPM.
- **Sharpe Ratio** measures return relative to total risk.
- **Sortino Ratio** focuses on downside risk.
- **Maximum Drawdown** shows the largest decline from a previous peak.
- **VaR** estimates a loss threshold at a chosen confidence level.
- **Expected Shortfall** estimates the average loss beyond the VaR threshold.

### Important

These results are based on historical data and should not be interpreted as a prediction of future returns or as investment advice.
